# Fig. 2k–l — Vulnerability index across VHC subtypes

Apply the vulnerability index (Fig 2h–j) to our in-house Smart-seq3 VHC dataset:

- **Fig 2k**: VHC-I vs VHC-II (two-group comparison)
- **Fig 2l**: All 9 VHC subtypes, ordered by median vulnerability (sorted descending)

## Scoring method differs from Fig 2i–j ⚠️

Public datasets (Fig 2i–j) used `geneset_mean` (raw log-normalised mean over the panel). Here we use **`sc.tl.score_genes`** which subtracts mean expression of a size-matched random reference set per cell. Rationale: within a single dataset, `sc.tl.score_genes` improves dynamic range and gives cleaner separation between fine subtypes, at the cost of producing signed scores that aren't directly comparable across datasets.

Because `score_genes` output can be negative, the `log2(...)` step needs a small positive offset (we use `eps = 1e-9`) and a small fraction of cells whose `proteostasis_capacity` happens to be near zero produce `vulnerability_log2ratio` outliers / NaN. We drop these via the `<= 15` cutoff used in Fig 2j.

## VHC-I / VHC-II assignment

| Cluster | Type |
| --- | --- |
| HC_Gsn, HC_Agbl1, HC_Socs3, HC_Car8, HC_Atp2a3, HC_Tshz3 | Type_I |
| HC_Foxp2, HC_Kcnv1, HC_St3gal6 | Type_II |

Hand-curated from the canonical Type I (calyx-bearing) vs Type II (bouton-bearing) markers (Sall3, Ocm, Spp1) and the VHC type annotation in `adata.obs['HC_Type']`.

## Inputs
- `Vestibular_VGN_VHC_merged.h5ad` — subset to `batch == 'HC_ss3'`

## Outputs
- `figures/Fig2k_VHC_TypeI_vs_TypeII.pdf` — Fig 2k boxplot
- `figures/Fig2l_VHC_subtypes_sorted.pdf` — Fig 2l boxplot (9 subtypes)
- `figures/Fig2kl_summary_stats.csv` — MWU statistics

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

from _gene_sets import metabolic_genes, proteostasis_genes

sc.settings.verbosity = 1
sc.set_figure_params(figsize=(5, 5), dpi_save=300, dpi=100, frameon=False)
mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.family'] = 'Arial'

DATA_DIR = Path('..')
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

EPS = 1e-9
VUL_CLIP_HIGH = 15  # drop pathological log2-ratio outliers

VHC_TYPE_MAP = {
    'HC_Gsn':     'Type_I',
    'HC_Agbl1':   'Type_I',
    'HC_Socs3':   'Type_I',
    'HC_Car8':    'Type_I',
    'HC_Atp2a3':  'Type_I',
    'HC_Tshz3':   'Type_I',
    'HC_Foxp2':   'Type_II',
    'HC_Kcnv1':   'Type_II',
    'HC_St3gal6': 'Type_II',
}

VHC_TYPE_PALETTE = {'Type_I': '#e74c3c', 'Type_II': '#57a65a'}

## 1. Load VHC subset, score gene panels with `sc.tl.score_genes`

In [ ]:
adata_all = sc.read_h5ad(DATA_DIR / 'Vestibular_VGN_VHC_merged.h5ad')
adata = adata_all[adata_all.obs['batch'].isin(['HC_ss3'])].copy()
adata.X = adata.layers['umi'].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
print(adata)
print('\nVHC subtypes:')
print(adata.obs['all_cluster_gene_name'].value_counts())

In [ ]:
# panel filtering
met_use = [g for g in metabolic_genes if g in adata.var_names]
pro_use = [g for g in proteostasis_genes if g in adata.var_names]
print(f'metabolic    used: {len(met_use)}/{len(metabolic_genes)}')
print(f'proteostasis used: {len(pro_use)}/{len(proteostasis_genes)}')

# sc.tl.score_genes: per-cell mean of panel minus mean of a size-matched random reference set
sc.tl.score_genes(adata, met_use, score_name='metabolic_load', random_state=0)
sc.tl.score_genes(adata, pro_use, score_name='proteostasis_capacity', random_state=0)

# log2 ratio — eps keeps the log defined for cells with near-zero proteostasis score
adata.obs['vulnerability_log2ratio'] = (
    np.log2(adata.obs['metabolic_load'] + EPS) -
    np.log2(adata.obs['proteostasis_capacity'] + EPS)
).replace([np.inf, -np.inf], np.nan)

# VHC-I / VHC-II annotation
adata.obs['VHCTypes'] = adata.obs['all_cluster_gene_name'].astype(str).map(VHC_TYPE_MAP)
adata.obs['VHCTypes'] = pd.Categorical(adata.obs['VHCTypes'], categories=['Type_I', 'Type_II'], ordered=True)

print('\nVulnerability score summary:')
print(adata.obs[['metabolic_load', 'proteostasis_capacity', 'vulnerability_log2ratio']].describe())

## 2. Fig 2k — Vulnerability index, VHC-I vs VHC-II

In [ ]:
plot_df = adata.obs.dropna(subset=['vulnerability_log2ratio', 'VHCTypes']).copy()
plot_df = plot_df[plot_df['vulnerability_log2ratio'] <= VUL_CLIP_HIGH]

# order x-axis by median (descending) so the higher-vulnerability group is on the left
ordered = (
    plot_df.groupby('VHCTypes', observed=True)['vulnerability_log2ratio']
    .median().sort_values(ascending=False).index.tolist()
)
plot_df['VHCTypes'] = pd.Categorical(plot_df['VHCTypes'], categories=ordered, ordered=True)

fig, ax = plt.subplots(figsize=(4, 5))
sns.boxplot(
    data=plot_df, x='VHCTypes', y='vulnerability_log2ratio',
    ax=ax, hue='VHCTypes', palette=VHC_TYPE_PALETTE,
    boxprops=dict(facecolor='none'), legend=False,
)
sns.stripplot(
    data=plot_df, x='VHCTypes', y='vulnerability_log2ratio',
    ax=ax, hue='VHCTypes', palette=VHC_TYPE_PALETTE, legend=False,
    size=2.5, alpha=0.6, jitter=0.2,
)

t1 = plot_df.loc[plot_df['VHCTypes'] == ordered[0], 'vulnerability_log2ratio']
t2 = plot_df.loc[plot_df['VHCTypes'] == ordered[1], 'vulnerability_log2ratio']
stat, pval = mannwhitneyu(t1, t2, alternative='two-sided')
n1, n2 = len(t1), len(t2)
rb = 1 - 2 * stat / (n1 * n2) if n1 * n2 > 0 else np.nan
ax.text(0.5, plot_df['vulnerability_log2ratio'].max() * 1.02,
        f'p = {pval:.2e}\n(MWU)',
        ha='center', va='bottom', transform=ax.get_xaxis_transform(), fontsize=9)

ax.set_xlabel('')
ax.set_ylabel('Vulnerability (log2 metabolic - proteostasis)')
ax.set_title('VHC-I vs VHC-II')
plt.tight_layout()
fig.savefig(FIG_DIR / 'Fig2k_VHC_TypeI_vs_TypeII.pdf', dpi=300, bbox_inches='tight')
plt.show()

stats_2k = {
    'panel': 'Fig 2k', 'dataset': 'in-house VHC',
    f'{ordered[0]}_n': n1, f'{ordered[0]}_median': float(t1.median()),
    f'{ordered[1]}_n': n2, f'{ordered[1]}_median': float(t2.median()),
    f'{ordered[0]}_minus_{ordered[1]}_median_diff': float(t1.median() - t2.median()),
    'MWU_U': float(stat), 'MWU_p': float(pval), 'rank_biserial_r': float(rb),
}
stats_2k

## 3. Fig 2l — Vulnerability index across 9 VHC subtypes (sorted by median)

In [ ]:
plot_df = adata.obs.dropna(subset=['vulnerability_log2ratio', 'all_cluster_gene_name']).copy()
plot_df = plot_df[plot_df['vulnerability_log2ratio'] <= VUL_CLIP_HIGH]

# x-axis ordered by per-cluster median vulnerability (descending)
ordered_clusters = (
    plot_df.groupby('all_cluster_gene_name', observed=True)['vulnerability_log2ratio']
    .median().sort_values(ascending=False).index.tolist()
)
plot_df['all_cluster_gene_name'] = pd.Categorical(
    plot_df['all_cluster_gene_name'], categories=ordered_clusters, ordered=True,
)

# adopt the canonical cluster palette from adata.uns (set during integration)
palette = None
if 'all_cluster_gene_name_colors' in adata.uns:
    cats = list(adata.obs['all_cluster_gene_name'].cat.categories)
    palette = dict(zip(cats, list(adata.uns['all_cluster_gene_name_colors'])))

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(
    data=plot_df, x='all_cluster_gene_name', y='vulnerability_log2ratio',
    ax=ax, hue='all_cluster_gene_name', palette=palette,
    boxprops=dict(facecolor='none'), legend=False,
)
sns.stripplot(
    data=plot_df, x='all_cluster_gene_name', y='vulnerability_log2ratio',
    ax=ax, hue='all_cluster_gene_name', palette=palette, legend=False,
    size=2.5, alpha=0.6, jitter=0.2,
)
ax.set_xlabel('')
ax.set_ylabel('Vulnerability (log2 metabolic - proteostasis)')
ax.set_title('VHC subtypes — vulnerability index (sorted by median)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
fig.savefig(FIG_DIR / 'Fig2l_VHC_subtypes_sorted.pdf', dpi=300, bbox_inches='tight')
plt.show()

# pairwise MWU: each subtype vs all-others (one-vs-rest) for transparency
rows_2l = []
all_vals = plot_df['vulnerability_log2ratio']
for cl in ordered_clusters:
    mask = (plot_df['all_cluster_gene_name'] == cl).values
    in_vals = plot_df.loc[mask, 'vulnerability_log2ratio']
    out_vals = plot_df.loc[~mask, 'vulnerability_log2ratio']
    stat, pval = mannwhitneyu(in_vals, out_vals, alternative='two-sided')
    n1, n2 = len(in_vals), len(out_vals)
    rb = 1 - 2 * stat / (n1 * n2) if n1 * n2 > 0 else np.nan
    rows_2l.append({
        'panel': 'Fig 2l (one-vs-rest)', 'cluster': cl,
        'n_in': n1, 'median_in': float(in_vals.median()),
        'n_out': n2, 'median_out': float(out_vals.median()),
        'median_diff_in_minus_out': float(in_vals.median() - out_vals.median()),
        'MWU_U': float(stat), 'MWU_p': float(pval), 'rank_biserial_r': float(rb),
    })

stats_2l_df = pd.DataFrame(rows_2l)
stats_2l_df

## 4. Export combined summary

In [ ]:
summary_df = pd.concat([pd.DataFrame([stats_2k]), stats_2l_df], ignore_index=True)
summary_df.to_csv(FIG_DIR / 'Fig2kl_summary_stats.csv', index=False)
summary_df